# Standalone Experiment C — Wavelet Frequency Benchmark

This notebook benchmarks the **marginal downstream utility of the fifth Custom Wavelet frequency** while keeping the four-channel backbone fixed at:

\[
(1, 2, 4, 8)\ \mathrm{Hz}
\]

For every Action 0 + Action 1 root pair, the notebook reads the encoder metadata, consumes one occurrence of each fixed backbone frequency `(1,2,4,8)`, and identifies the single remaining frequency as the **fifth channel**. This multiset rule intentionally allows duplicate controls such as `(1,1,2,4,8)` or `(1,2,2,4,8)`. The treatment is then named by that frequency (for example `0.5Hz`, `6Hz`, or `16Hz`).

Protocol:

- **No Experiment A checkpoint is required.**
- The **user split is fixed once** and reused by every run.
- Five **training/evaluation random seeds** are the replicate / variation source.
- Three matched CNN probes are evaluated separately: `cnn_s`, `cnn_m`, `cnn_l`.
- Every frequency condition must pass a strict **cross-combination preflight** before training.
- The standard reconstruction-only Experiment C evaluation is reused.
- The baseline/reference fifth channel defaults to **0.5 Hz**, corresponding to `(0.5, 1, 2, 4, 8)`. A sweep manifest may include this source baseline even when the generated sweep itself starts above 0.5 Hz.

Primary comparison unit:

\[
\text{fifth wavelet frequency} \times \text{CNN probe} \times \text{training seed}
\]

Main evidence hierarchy:

1. **Task performance:** CNN balanced accuracy and CNN macro-F1.
2. **Representation readout:** kNN balanced accuracy, linear-probe balanced accuracy, retrieval macro-mAP.
3. **Class-level behavior:** per-class recall heatmaps and confusion matrices.
4. **Supplementary geometry diagnostics:** SameLabel@1, intra/inter-class geometry, silhouette.

The notebook reports both absolute **mean ± sample SD across seeds** and seed-paired deltas relative to the reference fifth frequency.

## 1. Imports and repository root

Run this notebook from the repository checkout. The helper below finds the `writingRing` root when the notebook is opened from `notebooks/` or another subdirectory.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import json
import math
import re
import sys
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "snn").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the writingRing repository root. "
        "Open this notebook from inside the repository checkout."
    )


REPO_ROOT = find_repository_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval import (
    experiment_c_config,
    load_acceleration_data,
    prepare_user_disjoint_splits,
    restrict_manifest_to_cohort,
)
from snn.accel_reconstruction_eval.config import UserSplitConfig
from scripts.run_experiment_c import run_experiment_c

print("Repository root:", REPO_ROOT)

## 2. Benchmark configuration

Edit this cell before running the benchmark.

### Frequency treatment definition

`FIXED_WAVELET_CHANNELS_HZ = (1, 2, 4, 8)` defines the four backbone frequencies that must be present at least once in every condition. One occurrence of each is consumed; a duplicate occurrence may remain as the fifth-channel treatment.

You can either specify **Action 0 + Action 1 root pairs** in `COMBINATION_ROOTS`, or point `BENCHMARK_INPUT_MANIFEST` at the JSON produced by `sweep_fifth_wavelet_frequency.bash`. During preflight the notebook reads `spike_encoder.frequencies_hz`, consumes one occurrence of each fixed `(1,2,4,8)` backbone frequency, and uses the single remaining frequency as the condition name.

Examples:

- `(0.5, 1, 2, 4, 8)` → `0.5Hz`
- `(1, 2, 3, 4, 8)` → `3Hz`
- `(1, 1, 2, 4, 8)` → `1Hz` (duplicate control)
- `(1, 2, 2, 4, 8)` → `2Hz` (duplicate control)
- `(1, 2, 4, 8, 16)` → `16Hz`

The number of combinations is unrestricted. Frequency conditions are sorted numerically for plots and reports.

### Reference condition

`REFERENCE_FIFTH_CHANNEL_HZ = 0.5` defines the baseline for seed-paired delta analysis. The corresponding condition **must be present** in `COMBINATION_ROOTS`.

### Seeds

`SPLIT_SEED` is used once to create the fixed user-disjoint split from the reference condition. `TRAINING_SEEDS` are the repeated-training seeds and are the only variation summarized by the final SD/error bars.

### Label selection

`INCLUDED_LABELS` is applied after `EXCLUDED_USERS` and controls the final cohort used by preflight, every training split, and every test split. Set it to `None` to retain every label. Each requested label must survive user exclusion.

### Sweep manifest

When `BENCHMARK_INPUT_MANIFEST` is set, its `conditions` array supplies the root pairs automatically. The sweep launcher writes the original input combination as the baseline condition plus every generated fifth-frequency condition, so a sweep that does not itself include 0.5 Hz can still retain the 0.5 Hz reference.


In [ ]:
# -----------------------------------------------------------------------------
# USER CONFIGURATION
# -----------------------------------------------------------------------------

# Four wavelet frequencies that must be present in every condition.
FIXED_WAVELET_CHANNELS_HZ: tuple[float, ...] = (1.0, 2.0, 4.0, 8.0)

# Baseline fifth channel used for paired delta-vs-reference analysis.
# With the current baseline this corresponds to (0.5, 1, 2, 4, 8).
REFERENCE_FIFTH_CHANNEL_HZ: float = 0.5
FREQUENCY_MATCH_ATOL: float = 1e-9
FREQUENCY_AXIS_SCALE: str = "log"  # "log" is natural for multiplicative frequency spacing; "linear" is also valid.

# Optional JSON generated by sweep_fifth_wavelet_frequency.bash.
# When set, its condition root pairs replace the manual COMBINATION_ROOTS below.
BENCHMARK_INPUT_MANIFEST: Path | None = None
# Example:
# BENCHMARK_INPUT_MANIFEST = REPO_ROOT / "outputs/fifth_wavelet_frequency_sweep/experiment_c_frequency_benchmark_inputs.json"

# Manual fallback: add as many Action0 + Action1 root pairs as needed.
# Names are NOT supplied here; preflight derives each name from the fifth wavelet frequency.
COMBINATION_ROOTS: tuple[tuple[Path, Path], ...] = (
    (
        REPO_ROOT / "outputs/action0_rectified/low-pass/aligned-board-events",
        REPO_ROOT / "outputs/action1_rectified/low-pass/aligned-board-events",
    ),
)


def load_combination_roots_from_manifest(path: Path) -> tuple[tuple[Path, Path], ...]:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if payload.get("schema") != "writingring_experiment_c_frequency_benchmark_inputs_v1":
        raise ValueError(f"Unsupported benchmark input manifest schema in {path}: {payload.get('schema')!r}")
    conditions = payload.get("conditions")
    if not isinstance(conditions, list) or not conditions:
        raise ValueError(f"Benchmark input manifest has no conditions: {path}")
    roots: list[tuple[Path, Path]] = []
    for index, item in enumerate(conditions):
        if not isinstance(item, dict):
            raise ValueError(f"Manifest condition {index} is not an object")
        action0 = item.get("action0_root")
        action1 = item.get("action1_root")
        if not isinstance(action0, str) or not isinstance(action1, str):
            raise ValueError(f"Manifest condition {index} is missing action0_root/action1_root")
        roots.append((Path(action0), Path(action1)))
    return tuple(roots)


if BENCHMARK_INPUT_MANIFEST is not None:
    COMBINATION_ROOTS = load_combination_roots_from_manifest(BENCHMARK_INPUT_MANIFEST)

PROBES: tuple[str, ...] = ("cnn_s", "cnn_m", "cnn_l")
TRAINING_SEEDS: tuple[int, ...] = (13, 37, 71, 101, 137)
SPLIT_SEED: int = 12345

# Cohort controls applied before the fixed split is created.
EXCLUDED_USERS: tuple[str, ...] = ("user_17",)
# Ordered labels to retain after user exclusion; None keeps every label.
INCLUDED_LABELS: tuple[str, ...] | None = ("A", "B", "C", "D", "E", "X", "G", "H", "I", "J", "K", "L")
TRAIN_FRACTION: float = 0.70
VAL_FRACTION: float = 0.15
REQUIRE_ALL_LABELS_IN_ALL_SPLITS: bool = True

# -----------------------------------------------------------------------------
# METRICS
# -----------------------------------------------------------------------------

# Highest-priority downstream task metrics.
TASK_METRICS: dict[str, str] = {
    "CNN balanced accuracy": "CNN_test_balanced_accuracy",
    "CNN macro-F1": "CNN_test_macro_f1",
}

# Representation-level evidence that class information remains directly readable.
REPRESENTATION_METRICS: dict[str, str] = {
    "kNN balanced accuracy": "kNN_test_balanced_accuracy",
    "Linear probe balanced accuracy": "linear_probe_test_balanced_accuracy",
    "Retrieval macro mAP": "retrieval_macro_mAP",
}

# These five metrics receive the main frequency-response and paired-delta figures.
SELECTED_METRICS: dict[str, str] = {
    **TASK_METRICS,
    **REPRESENTATION_METRICS,
}

# Saved as supporting diagnostics, but not treated as primary frequency-ranking criteria.
SUPPLEMENTARY_METRICS: dict[str, str] = {
    "SameLabel@1 macro": "SameLabel_macro_at_1",
    "D_intra macro": "D_intra_macro",
    "D_inter centroids": "D_inter_centroids",
    "D_inter / D_intra": "D_inter_over_D_intra",
    "Silhouette macro": "silhouette_macro",
}

ALL_REPORTED_METRICS: dict[str, str] = {
    **SELECTED_METRICS,
    **SUPPLEMENTARY_METRICS,
}

OUTPUT_ROOT = REPO_ROOT / "notebooks/artifacts/experiment_C_frequency_benchmark"
FIGURE_ROOT = OUTPUT_ROOT / "comparison_figures"

# None = let run_experiment_c choose CUDA when configured/available.
DEVICE: str | None = None

# If True, existing run directories with summary.csv + embeddings_test.npz are reused.
RESUME: bool = True

# Expensive but strongest preflight: require raw IMU channels 15:21 to be exactly
# identical across encoder conditions, package by package. Event channels 0:15
# are intentionally allowed to differ.
STRICT_RAW_IMU_EQUAL: bool = True
RAW_IMU_COMPARE_SEGMENT_CHUNK: int = 64

# Set True only when you are ready to launch all trainings.
RUN_BENCHMARK: bool = False

# Visualization controls.
ANNOTATE_CONFUSION_IF_CLASSES_LEQ: int = 12
PLOT_CONFUSION_STD: bool = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

print("Fixed wavelet backbone (Hz):", FIXED_WAVELET_CHANNELS_HZ)
print("Reference fifth channel (Hz):", REFERENCE_FIFTH_CHANNEL_HZ)
print("Included labels:", INCLUDED_LABELS)
print("Benchmark input manifest:", BENCHMARK_INPUT_MANIFEST)
print("Root-pair count:", len(COMBINATION_ROOTS))
print("Training seeds:", TRAINING_SEEDS)
print("Probes:", PROBES)
print("Output root:", OUTPUT_ROOT)

## 3. Cross-combination preflight and fixed split

This preflight enforces both the original cohort controls and the new frequency-sweep treatment definition.

For every Action0/Action1 root pair it checks:

- exactly two roots are supplied;
- reconstruction artifacts load and validate;
- both roots declare the same five `frequencies_hz` values;
- the encoder contains each fixed `(1,2,4,8)` Hz backbone frequency at least once;
- after consuming one occurrence of each backbone frequency, exactly one remaining frequency can be identified as the fifth channel;
- duplicate fifth-channel controls such as `(1,1,2,4,8)` are allowed;
- fifth-channel condition names are unique;
- the configured reference fifth frequency (default `0.5 Hz`) is present;
- the selected Action set, target length, and sampling rate match the reference;
- canonical sample IDs are identical after applying the fixed cohort;
- user/action/label and `valid_length` agree sample-by-sample;
- package keys, labels, valid lengths, and valid masks agree;
- optionally, SpikeIMU channels `15:21` are exactly equal across conditions.

The last check verifies that copied raw IMU channels did not change while event channels `0:15` are allowed to differ. If any assumption fails, the benchmark stops rather than silently intersecting or relabeling datasets.

In [ ]:
def safe_slug(text: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    if not slug:
        raise ValueError(f"Condition name cannot be converted to a safe path: {text!r}")
    return slug


def frequency_label(frequency_hz: float) -> str:
    return f"{float(frequency_hz):g}Hz"


def _frequency_matches(left: float, right: float) -> bool:
    return bool(np.isclose(float(left), float(right), rtol=0.0, atol=FREQUENCY_MATCH_ATOL))


def _extract_wavelet_frequencies(data: object) -> tuple[float, ...]:
    per_root: list[tuple[float, ...]] = []
    for root_index, metadata in enumerate(data.producer_metadatas):
        spec = metadata.spike_encoder_spec
        if not isinstance(spec, dict):
            raise ValueError(
                f"Root {root_index} does not expose spike_encoder metadata; "
                "cannot infer the fifth wavelet channel"
            )
        values = spec.get("frequencies_hz")
        if values is None:
            raise ValueError(
                f"Root {root_index} spike_encoder metadata is missing frequencies_hz"
            )
        try:
            frequencies = tuple(float(value) for value in values)
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"Root {root_index} has invalid frequencies_hz: {values!r}"
            ) from exc
        if len(frequencies) != 5:
            raise ValueError(
                f"Expected exactly five wavelet frequencies, got {frequencies!r}"
            )
        per_root.append(frequencies)

    if not per_root:
        raise ValueError("Loaded condition has no producer metadata")

    reference = per_root[0]
    for root_index, frequencies in enumerate(per_root[1:], start=1):
        if len(reference) != len(frequencies) or not all(
            _frequency_matches(left, right)
            for left, right in zip(reference, frequencies)
        ):
            raise AssertionError(
                "Action0/Action1 encoder frequency mismatch: "
                f"{reference!r} vs {frequencies!r} (root index {root_index})"
            )
    return reference


def _infer_fifth_channel(frequencies_hz: Sequence[float]) -> float:
    """Consume one occurrence of each fixed backbone frequency and return the remainder.

    This is intentionally multiset-based so duplicate fifth-channel controls are valid:
    (1, 1, 2, 4, 8) -> 1 Hz, (1, 2, 2, 4, 8) -> 2 Hz, etc.
    """
    remaining = [float(value) for value in frequencies_hz]
    for fixed in FIXED_WAVELET_CHANNELS_HZ:
        matches = [
            index
            for index, value in enumerate(remaining)
            if _frequency_matches(value, fixed)
        ]
        if not matches:
            raise ValueError(
                "Every condition must contain each fixed backbone frequency at least once. "
                f"Missing {fixed:g} Hz in {tuple(frequencies_hz)!r}"
            )
        remaining.pop(matches[0])

    if len(remaining) != 1:
        raise ValueError(
            "Could not identify exactly one fifth wavelet channel after consuming one occurrence "
            f"of each fixed backbone frequency {FIXED_WAVELET_CHANNELS_HZ}: {tuple(frequencies_hz)!r}"
        )
    return float(remaining[0])


def _semantic_manifest(frame: pd.DataFrame) -> pd.DataFrame:
    required = ["sample_id", "user", "action", "label"]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f"sample_manifest is missing required columns: {missing}")

    columns = required.copy()
    if "valid_length" in frame.columns:
        columns.append("valid_length")

    result = frame.loc[:, columns].copy()
    result["sample_id"] = result["sample_id"].astype(str)
    result["user"] = result["user"].astype(str)
    result["action"] = result["action"].astype(str)
    result["label"] = result["label"].astype(str)
    result = result.sort_values("sample_id", kind="stable").reset_index(drop=True)
    if result["sample_id"].duplicated().any():
        duplicates = result.loc[result["sample_id"].duplicated(), "sample_id"].tolist()
        raise ValueError(f"Duplicate sample IDs found: {duplicates[:10]}")
    return result


def _package_map(data) -> dict[tuple[str, str], object]:
    mapping: dict[tuple[str, str], object] = {}
    for package in data.packages:
        key = (str(package.user), str(package.action))
        if key in mapping:
            raise ValueError(f"Duplicate package key: {key}")
        mapping[key] = package
    return mapping


def _assert_raw_imu_equal(reference_package, current_package, chunk_segments: int) -> None:
    ref = reference_package.padded_spike_imu
    cur = current_package.padded_spike_imu
    if ref.shape != cur.shape:
        raise AssertionError(
            f"Raw SpikeIMU shape mismatch for {(reference_package.user, reference_package.action)}: "
            f"{ref.shape} vs {cur.shape}"
        )
    for start in range(0, ref.shape[0], chunk_segments):
        stop = min(start + chunk_segments, ref.shape[0])
        if not np.array_equal(ref[start:stop, :, 15:21], cur[start:stop, :, 15:21]):
            raise AssertionError(
                "Raw IMU channels 15:21 differ for package "
                f"{(reference_package.user, reference_package.action)} "
                f"at segment slice [{start}:{stop}]"
            )


def run_preflight(
    combination_roots: Sequence[Sequence[Path]],
) -> tuple[
    dict[str, tuple[Path, Path]],
    dict[str, object],
    object,
    UserSplitConfig,
    pd.DataFrame,
    dict[str, float],
    str,
]:
    if not combination_roots:
        raise ValueError("COMBINATION_ROOTS must contain at least one Action0/Action1 pair")

    candidates: list[dict[str, object]] = []
    for index, roots_value in enumerate(combination_roots):
        roots = tuple(roots_value)
        if len(roots) != 2:
            raise ValueError(
                f"COMBINATION_ROOTS[{index}] must contain exactly two roots: Action0 then Action1"
            )
        roots = (Path(roots[0]), Path(roots[1]))
        data = load_acceleration_data(
            list(roots),
            repository_root=REPO_ROOT,
            require_reconstruction=True,
        )
        frequencies = _extract_wavelet_frequencies(data)
        fifth_channel = _infer_fifth_channel(frequencies)
        name = frequency_label(fifth_channel)
        candidates.append(
            {
                "name": name,
                "fifth_channel_hz": fifth_channel,
                "frequencies_hz": frequencies,
                "roots": roots,
                "data": data,
            }
        )

    # Conditions are identified by the numeric fifth frequency, not by user-entered labels.
    for left_index, left in enumerate(candidates):
        for right in candidates[left_index + 1:]:
            if _frequency_matches(left["fifth_channel_hz"], right["fifth_channel_hz"]):
                raise ValueError(
                    "Duplicate fifth-channel condition detected: "
                    f"{left['fifth_channel_hz']:g} Hz. Each frequency must appear once."
                )

    candidates.sort(key=lambda item: float(item["fifth_channel_hz"]))
    combinations = {
        str(item["name"]): item["roots"]
        for item in candidates
    }
    loaded = {
        str(item["name"]): item["data"]
        for item in candidates
    }
    fifth_channel_by_name = {
        str(item["name"]): float(item["fifth_channel_hz"])
        for item in candidates
    }
    frequencies_by_name = {
        str(item["name"]): tuple(item["frequencies_hz"])
        for item in candidates
    }

    reference_matches = [
        name
        for name, fifth_channel in fifth_channel_by_name.items()
        if _frequency_matches(fifth_channel, REFERENCE_FIFTH_CHANNEL_HZ)
    ]
    if len(reference_matches) != 1:
        raise ValueError(
            "The configured reference fifth channel must be present exactly once. "
            f"reference={REFERENCE_FIFTH_CHANNEL_HZ:g} Hz, available={list(fifth_channel_by_name.values())}"
        )
    reference_name = reference_matches[0]
    reference_data = loaded[reference_name]

    reference_split = prepare_user_disjoint_splits(
        reference_data.sample_manifest,
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SPLIT_SEED,
        excluded_users=EXCLUDED_USERS,
        included_labels=INCLUDED_LABELS,
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
    )
    if INCLUDED_LABELS is not None:
        expected_labels = {str(label) for label in INCLUDED_LABELS}
        actual_labels = set(reference_split.sample_manifest["label"].astype(str))
        if actual_labels != expected_labels:
            raise AssertionError(
                f"Reference split labels {sorted(actual_labels)} do not match "
                f"INCLUDED_LABELS {list(INCLUDED_LABELS)}"
            )

    fixed_split_config = UserSplitConfig(
        explicit_train_users=tuple(reference_split.train_users),
        explicit_val_users=tuple(reference_split.val_users),
        explicit_test_users=tuple(reference_split.test_users),
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
        excluded_users=EXCLUDED_USERS,
        included_labels=INCLUDED_LABELS,
    )
    fixed_split_config.validate()

    split_users = (
        *reference_split.train_users,
        *reference_split.val_users,
        *reference_split.test_users,
    )
    canonical = restrict_manifest_to_cohort(
        reference_data.sample_manifest,
        split_users=split_users,
        class_to_idx=reference_split.class_to_idx,
    )
    canonical_semantic = _semantic_manifest(canonical)
    canonical_ids = tuple(canonical_semantic["sample_id"].tolist())

    reference_actions = tuple(reference_data.selected_actions)
    reference_target_length = int(reference_data.producer_metadata.target_length)
    reference_sampling_rate = float(reference_data.producer_metadata.sampling_rate_hz)
    reference_packages = _package_map(reference_data)

    rows: list[dict[str, object]] = []

    for name, data in loaded.items():
        current = restrict_manifest_to_cohort(
            data.sample_manifest,
            split_users=split_users,
            class_to_idx=reference_split.class_to_idx,
        )
        current_semantic = _semantic_manifest(current)
        current_ids = tuple(current_semantic["sample_id"].tolist())

        if current_ids != canonical_ids:
            reference_set = set(canonical_ids)
            current_set = set(current_ids)
            missing = sorted(reference_set - current_set)[:20]
            extra = sorted(current_set - reference_set)[:20]
            raise AssertionError(
                f"Cross-combination cohort mismatch for {name!r}. "
                f"Missing IDs (first 20): {missing}; extra IDs (first 20): {extra}"
            )

        compare_columns = [
            column
            for column in ("sample_id", "user", "action", "label", "valid_length")
            if column in canonical_semantic.columns and column in current_semantic.columns
        ]
        if not canonical_semantic[compare_columns].equals(current_semantic[compare_columns]):
            raise AssertionError(
                f"Semantic manifest mismatch for {name!r} in columns {compare_columns}"
            )

        actions = tuple(data.selected_actions)
        if actions != reference_actions:
            raise AssertionError(
                f"Action-set mismatch for {name!r}: {actions} vs {reference_actions}"
            )
        if int(data.producer_metadata.target_length) != reference_target_length:
            raise AssertionError(
                f"target_length mismatch for {name!r}: "
                f"{data.producer_metadata.target_length} vs {reference_target_length}"
            )
        if not np.isclose(float(data.producer_metadata.sampling_rate_hz), reference_sampling_rate):
            raise AssertionError(
                f"sampling_rate_hz mismatch for {name!r}: "
                f"{data.producer_metadata.sampling_rate_hz} vs {reference_sampling_rate}"
            )

        current_packages = _package_map(data)
        if current_packages.keys() != reference_packages.keys():
            raise AssertionError(f"Package-key mismatch for {name!r}")

        if name != reference_name:
            for key in reference_packages:
                ref_package = reference_packages[key]
                cur_package = current_packages[key]
                if not np.array_equal(ref_package.labels, cur_package.labels):
                    raise AssertionError(f"Label-array mismatch for {name!r}, package {key}")
                if not np.array_equal(ref_package.valid_lengths, cur_package.valid_lengths):
                    raise AssertionError(f"valid_lengths mismatch for {name!r}, package {key}")
                if not np.array_equal(ref_package.valid_mask, cur_package.valid_mask):
                    raise AssertionError(f"valid_mask mismatch for {name!r}, package {key}")
                if STRICT_RAW_IMU_EQUAL:
                    _assert_raw_imu_equal(
                        ref_package,
                        cur_package,
                        chunk_segments=RAW_IMU_COMPARE_SEGMENT_CHUNK,
                    )

        encoder_hashes = sorted(
            {
                str(meta.spike_encoder_spec_sha256)
                for meta in data.producer_metadatas
                if meta.spike_encoder_spec_sha256 is not None
            }
        )
        rows.append(
            {
                "combination": name,
                "fifth_channel_hz": fifth_channel_by_name[name],
                "is_reference": name == reference_name,
                "fixed_wavelet_channels_hz": ",".join(f"{value:g}" for value in FIXED_WAVELET_CHANNELS_HZ),
                "all_wavelet_frequencies_hz": ",".join(f"{value:g}" for value in frequencies_by_name[name]),
                "root_count": len(data.padded_roots),
                "actions": ",".join(actions),
                "selected_samples": len(current_semantic),
                "selected_users": len(set(current_semantic["user"])),
                "selected_labels": len(set(current_semantic["label"])),
                "target_length": int(data.producer_metadata.target_length),
                "sampling_rate_hz": float(data.producer_metadata.sampling_rate_hz),
                "encoder_spec_hashes": " | ".join(encoder_hashes),
                "strict_raw_imu_equal_checked": bool(STRICT_RAW_IMU_EQUAL),
            }
        )

    preflight_table = pd.DataFrame(rows).sort_values("fifth_channel_hz").reset_index(drop=True)
    return (
        combinations,
        loaded,
        reference_split,
        fixed_split_config,
        preflight_table,
        fifth_channel_by_name,
        reference_name,
    )


(
    COMBINATIONS,
    LOADED_COMBINATIONS,
    FIXED_SPLIT,
    FIXED_SPLIT_CONFIG,
    PREFLIGHT_TABLE,
    FIFTH_CHANNEL_HZ_BY_COMBINATION,
    REFERENCE_COMBINATION,
) = run_preflight(COMBINATION_ROOTS)

COMBINATION_ORDER = list(COMBINATIONS)
FIFTH_CHANNEL_ORDER_HZ = [FIFTH_CHANNEL_HZ_BY_COMBINATION[name] for name in COMBINATION_ORDER]

display(PREFLIGHT_TABLE)
print("Derived conditions:", COMBINATION_ORDER)
print("Reference condition:", REFERENCE_COMBINATION)
print("Fixed train users:", FIXED_SPLIT.train_users)
print("Fixed val users:", FIXED_SPLIT.val_users)
print("Fixed test users:", FIXED_SPLIT.test_users)
print("Class mapping:", FIXED_SPLIT.class_to_idx)

## 4. Persist the benchmark design

The design file records the fixed backbone, automatically inferred fifth-channel frequencies, root pairs, fixed user split, seeds, probes, and metric hierarchy before training starts. This makes every frequency-response result auditable.

In [ ]:
DESIGN_PATH = OUTPUT_ROOT / "benchmark_design.json"
PREFLIGHT_CSV = OUTPUT_ROOT / "cross_combination_preflight.csv"

benchmark_design = {
    "protocol": "standalone_experiment_c_wavelet_frequency_benchmark_v3",
    "benchmark_input_manifest": None if BENCHMARK_INPUT_MANIFEST is None else str(Path(BENCHMARK_INPUT_MANIFEST).resolve()),
    "fixed_wavelet_channels_hz": list(FIXED_WAVELET_CHANNELS_HZ),
    "reference_fifth_channel_hz": float(REFERENCE_FIFTH_CHANNEL_HZ),
    "reference_combination": REFERENCE_COMBINATION,
    "combinations": {
        name: {
            "fifth_channel_hz": FIFTH_CHANNEL_HZ_BY_COMBINATION[name],
            "roots": [str(Path(root).resolve()) for root in roots],
        }
        for name, roots in COMBINATIONS.items()
    },
    "split_seed": SPLIT_SEED,
    "training_seeds": list(TRAINING_SEEDS),
    "probes": list(PROBES),
    "excluded_users": list(EXCLUDED_USERS),
    "included_labels": None if INCLUDED_LABELS is None else list(INCLUDED_LABELS),
    "train_users": list(FIXED_SPLIT.train_users),
    "val_users": list(FIXED_SPLIT.val_users),
    "test_users": list(FIXED_SPLIT.test_users),
    "class_to_idx": FIXED_SPLIT.class_to_idx,
    "task_metrics": TASK_METRICS,
    "representation_metrics": REPRESENTATION_METRICS,
    "supplementary_metrics": SUPPLEMENTARY_METRICS,
    "strict_raw_imu_equal": STRICT_RAW_IMU_EQUAL,
}

DESIGN_PATH.write_text(json.dumps(benchmark_design, indent=2), encoding="utf-8")
PREFLIGHT_TABLE.to_csv(PREFLIGHT_CSV, index=False)
print("Saved:", DESIGN_PATH)
print("Saved:", PREFLIGHT_CSV)

## 5. Run all standalone Experiment C trainings

Every run uses:

- `reference_checkpoint=None`;
- `allow_new_split=True`;
- the same explicit train/validation/test user lists;
- reconstruction-only train/validation/test;
- reconstruction-train normalization;
- one of the three probe variants;
- one of the five training/evaluation seeds.

Output directories are automatically named by the inferred fifth wavelet frequency:

```text
notebooks/artifacts/experiment_C_frequency_benchmark/
  0.5Hz/
    seed_<seed>/
      cnn_s/
      cnn_m/
      cnn_l/
  6Hz/
    ...
  16Hz/
    ...
```

`run_experiment_c()` appends the probe directory itself, so each standard C artifact layout remains intact.

In [ ]:
MASTER_RESULTS_PATH = OUTPUT_ROOT / "master_results.csv"


def build_c_config(seed: int, probe: str, output_base: Path):
    config = experiment_c_config(
        output_dir=output_base,
        random_seed=int(seed),
        probe_variant=probe,
    )
    config = replace(config, split=FIXED_SPLIT_CONFIG)
    config.validate()
    return config


def expected_run_dir(combination: str, seed: int, probe: str) -> Path:
    return OUTPUT_ROOT / safe_slug(combination) / f"seed_{seed}" / probe


def load_existing_summary(run_dir: Path) -> pd.DataFrame | None:
    path = run_dir / "summary.csv"
    if not path.is_file():
        return None
    if INCLUDED_LABELS is not None:
        manifest_path = run_dir / "sample_manifest.csv"
        if not manifest_path.is_file():
            return None
        manifest = pd.read_csv(manifest_path)
        if not {"label", "split", "user"}.issubset(manifest.columns):
            return None
        expected_labels = {str(label) for label in INCLUDED_LABELS}
        actual_labels = set(manifest["label"].astype(str))
        if actual_labels != expected_labels:
            return None
        expected_users_by_split = {
            "train": {str(user) for user in FIXED_SPLIT.train_users},
            "val": {str(user) for user in FIXED_SPLIT.val_users},
            "test": {str(user) for user in FIXED_SPLIT.test_users},
        }
        for split_name, expected_users in expected_users_by_split.items():
            actual_users = set(
                manifest.loc[manifest["split"] == split_name, "user"].astype(str)
            )
            if actual_users != expected_users:
                return None
    frame = pd.read_csv(path)
    if len(frame) != 1:
        raise ValueError(f"Expected one summary row in {path}, found {len(frame)}")
    return frame


def run_benchmark() -> pd.DataFrame:
    records: list[dict[str, object]] = []
    total = len(COMBINATIONS) * len(PROBES) * len(TRAINING_SEEDS)
    ordinal = 0

    for combination, roots in COMBINATIONS.items():
        combination_base = OUTPUT_ROOT / safe_slug(combination)
        fifth_channel_hz = FIFTH_CHANNEL_HZ_BY_COMBINATION[combination]
        for probe in PROBES:
            for seed in TRAINING_SEEDS:
                ordinal += 1
                run_base = combination_base / f"seed_{seed}"
                run_dir = run_base / probe
                summary_path = run_dir / "summary.csv"
                embedding_path = run_dir / "embeddings_test.npz"

                print(
                    f"[{ordinal}/{total}] fifth={fifth_channel_hz:g} Hz "
                    f"| {probe} | seed={seed}"
                )

                summary = None
                if RESUME and summary_path.is_file() and embedding_path.is_file():
                    summary = load_existing_summary(run_dir)
                if summary is None:
                    config = build_c_config(seed, probe, run_base)
                    result = run_experiment_c(
                        root=list(roots),
                        repository_root=REPO_ROOT,
                        output_dir=run_base,
                        reference_checkpoint=None,
                        config=config,
                        device=DEVICE,
                        allow_new_split=True,
                        probe_variant=probe,
                    )
                    summary = result.summary.copy()

                row = summary.iloc[0].to_dict()
                row.update(
                    {
                        "combination": combination,
                        "fifth_channel_hz": float(fifth_channel_hz),
                        "is_reference": combination == REFERENCE_COMBINATION,
                        "combination_slug": safe_slug(combination),
                        "probe": probe,
                        "training_seed": int(seed),
                        "run_dir": str(run_dir),
                    }
                )
                records.append(row)

                # Incremental checkpoint of benchmark progress.
                pd.DataFrame(records).to_csv(MASTER_RESULTS_PATH, index=False)

    result_frame = pd.DataFrame(records)
    result_frame.to_csv(MASTER_RESULTS_PATH, index=False)
    return result_frame


if RUN_BENCHMARK:
    MASTER_RESULTS = run_benchmark()
else:
    if INCLUDED_LABELS is not None:
        MASTER_RESULTS = pd.DataFrame()
        print("RUN_BENCHMARK=False; existing results skipped because label selection is active; set RUN_BENCHMARK=True to run or resume the selected cohort")
    elif MASTER_RESULTS_PATH.is_file():
        MASTER_RESULTS = pd.read_csv(MASTER_RESULTS_PATH)
        print("RUN_BENCHMARK=False; loaded existing master results:", MASTER_RESULTS_PATH)
    else:
        MASTER_RESULTS = pd.DataFrame()
        print(
            "RUN_BENCHMARK=False and no master_results.csv exists yet. "
            "Set RUN_BENCHMARK=True in the configuration cell when ready."
        )

display(MASTER_RESULTS.head() if not MASTER_RESULTS.empty else MASTER_RESULTS)

## 6. Aggregate metrics across training seeds

### Main metrics

The main frequency-response analysis contains:

- CNN balanced accuracy;
- CNN macro-F1;
- kNN balanced accuracy;
- linear-probe balanced accuracy;
- retrieval macro-mAP.

For every **fifth frequency × probe × metric**, report mean, sample SD (`ddof=1`), and completed-seed count. Probes remain separate.

### Paired delta vs 0.5 Hz reference

Because every condition uses the same five training-seed IDs, the notebook also computes seed-matched differences:

\[
\Delta M_s(f) = M_s(f) - M_s(f_{ref})
\]

and reports mean ± SD of those paired deltas. This is the most direct summary of marginal improvement relative to the baseline fifth frequency.

### Supplementary metrics

SameLabel@1 and geometry diagnostics are still saved for mechanism/stability interpretation but are not used as the primary frequency-ranking criteria.

In [ ]:
def require_complete_master_results(frame: pd.DataFrame) -> None:
    if frame.empty:
        raise RuntimeError("No benchmark results are available. Run the training cell first.")

    required = {
        "combination",
        "fifth_channel_hz",
        "probe",
        "training_seed",
        *ALL_REPORTED_METRICS.values(),
    }
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise KeyError(f"master_results.csv is missing required columns: {missing}")

    expected = {
        (combination, probe, int(seed))
        for combination in COMBINATIONS
        for probe in PROBES
        for seed in TRAINING_SEEDS
    }
    observed = {
        (str(row.combination), str(row.probe), int(row.training_seed))
        for row in frame.loc[:, ["combination", "probe", "training_seed"]].itertuples(index=False)
    }
    missing_runs = sorted(expected - observed)
    if missing_runs:
        raise RuntimeError(
            f"Benchmark is incomplete; missing {len(missing_runs)} runs: {missing_runs[:20]}"
        )


def metrics_to_long(frame: pd.DataFrame, metrics: Mapping[str, str]) -> pd.DataFrame:
    long_frames: list[pd.DataFrame] = []
    for display_name, column in metrics.items():
        part = frame.loc[
            :, ["combination", "fifth_channel_hz", "probe", "training_seed", column]
        ].copy()
        part = part.rename(columns={column: "value"})
        part["metric"] = display_name
        part["metric_column"] = column
        long_frames.append(part)
    return pd.concat(long_frames, ignore_index=True)


def aggregate_metric_long(long_table: pd.DataFrame) -> pd.DataFrame:
    aggregate = (
        long_table
        .groupby(
            ["metric", "metric_column", "combination", "fifth_channel_hz", "probe"],
            as_index=False,
        )
        .agg(mean=("value", "mean"), sd=("value", "std"), n=("value", "count"))
        .sort_values(["metric", "probe", "fifth_channel_hz"])
        .reset_index(drop=True)
    )
    return aggregate


def build_paired_deltas(long_table: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    reference = (
        long_table.loc[
            long_table["combination"] == REFERENCE_COMBINATION,
            ["metric", "metric_column", "probe", "training_seed", "value"],
        ]
        .rename(columns={"value": "reference_value"})
    )
    paired = long_table.merge(
        reference,
        on=["metric", "metric_column", "probe", "training_seed"],
        how="left",
        validate="many_to_one",
    )
    if paired["reference_value"].isna().any():
        raise RuntimeError("Reference condition is missing seed/probe/metric values for paired analysis")

    paired["delta_vs_reference"] = paired["value"] - paired["reference_value"]
    paired["reference_combination"] = REFERENCE_COMBINATION
    paired["reference_fifth_channel_hz"] = float(REFERENCE_FIFTH_CHANNEL_HZ)

    aggregate = (
        paired
        .groupby(
            ["metric", "metric_column", "combination", "fifth_channel_hz", "probe"],
            as_index=False,
        )
        .agg(
            mean_delta=("delta_vs_reference", "mean"),
            sd_delta=("delta_vs_reference", "std"),
            n=("delta_vs_reference", "count"),
        )
        .sort_values(["metric", "probe", "fifth_channel_hz"])
        .reset_index(drop=True)
    )
    return paired, aggregate


if not MASTER_RESULTS.empty:
    require_complete_master_results(MASTER_RESULTS)

    METRIC_LONG = metrics_to_long(MASTER_RESULTS, SELECTED_METRICS)
    METRIC_AGGREGATE = aggregate_metric_long(METRIC_LONG)
    METRIC_AGGREGATE.to_csv(OUTPUT_ROOT / "selected_metric_aggregate.csv", index=False)

    SUPPLEMENTARY_METRIC_LONG = metrics_to_long(MASTER_RESULTS, SUPPLEMENTARY_METRICS)
    SUPPLEMENTARY_METRIC_AGGREGATE = aggregate_metric_long(SUPPLEMENTARY_METRIC_LONG)
    SUPPLEMENTARY_METRIC_AGGREGATE.to_csv(
        OUTPUT_ROOT / "supplementary_metric_aggregate.csv", index=False
    )

    PAIRED_DELTA_BY_SEED, PAIRED_DELTA_AGGREGATE = build_paired_deltas(METRIC_LONG)
    PAIRED_DELTA_BY_SEED.to_csv(OUTPUT_ROOT / "paired_delta_by_seed.csv", index=False)
    PAIRED_DELTA_AGGREGATE.to_csv(OUTPUT_ROOT / "paired_delta_aggregate.csv", index=False)

    print("Main metric aggregate")
    display(METRIC_AGGREGATE)
    print("Paired deltas vs reference")
    display(PAIRED_DELTA_AGGREGATE)
    print("Supplementary metric aggregate")
    display(SUPPLEMENTARY_METRIC_AGGREGATE)
else:
    METRIC_LONG = pd.DataFrame()
    METRIC_AGGREGATE = pd.DataFrame()
    SUPPLEMENTARY_METRIC_LONG = pd.DataFrame()
    SUPPLEMENTARY_METRIC_AGGREGATE = pd.DataFrame()
    PAIRED_DELTA_BY_SEED = pd.DataFrame()
    PAIRED_DELTA_AGGREGATE = pd.DataFrame()

In [ ]:
def _configure_frequency_axis(ax: plt.Axes) -> None:
    if FREQUENCY_AXIS_SCALE not in {"linear", "log"}:
        raise ValueError("FREQUENCY_AXIS_SCALE must be 'linear' or 'log'")
    ax.set_xscale(FREQUENCY_AXIS_SCALE)
    ax.set_xticks(FIFTH_CHANNEL_ORDER_HZ)
    ax.set_xticklabels([f"{value:g}" for value in FIFTH_CHANNEL_ORDER_HZ])
    ax.set_xlabel("Fifth wavelet frequency (Hz)")


def plot_selected_metric_comparisons(
    aggregate: pd.DataFrame,
    *,
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if aggregate.empty:
        raise RuntimeError("No aggregate metrics to plot")

    output_dir.mkdir(parents=True, exist_ok=True)
    saved: dict[str, Path] = {}

    for metric_name in SELECTED_METRICS:
        subset = aggregate.loc[aggregate["metric"] == metric_name].copy()
        fig, ax = plt.subplots(figsize=(9.5, 5.4))

        for probe in PROBES:
            probe_table = (
                subset.loc[subset["probe"] == probe]
                .set_index("combination")
                .reindex(COMBINATION_ORDER)
            )
            if probe_table["mean"].isna().any():
                raise ValueError(f"Missing aggregate rows for {metric_name!r}, probe={probe!r}")

            ax.errorbar(
                FIFTH_CHANNEL_ORDER_HZ,
                probe_table["mean"].to_numpy(dtype=float),
                yerr=probe_table["sd"].to_numpy(dtype=float),
                marker="o",
                capsize=4,
                label=probe,
            )

        ax.axvline(
            REFERENCE_FIFTH_CHANNEL_HZ,
            linestyle="--",
            linewidth=1.2,
            label=f"reference = {REFERENCE_FIFTH_CHANNEL_HZ:g} Hz",
        )
        _configure_frequency_axis(ax)
        ax.set_ylabel(metric_name)
        ax.set_title(
            f"{metric_name}: fixed backbone {FIXED_WAVELET_CHANNELS_HZ} Hz, "
            f"mean ± SD across {len(TRAINING_SEEDS)} seeds"
        )
        ax.grid(alpha=0.25)
        ax.legend(title="CNN probe")
        fig.tight_layout()

        path = output_dir / f"metric_{safe_slug(metric_name)}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[metric_name] = path

    return saved


def plot_paired_delta_comparisons(
    aggregate: pd.DataFrame,
    *,
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if aggregate.empty:
        raise RuntimeError("No paired delta metrics to plot")

    output_dir.mkdir(parents=True, exist_ok=True)
    saved: dict[str, Path] = {}

    for metric_name in SELECTED_METRICS:
        subset = aggregate.loc[aggregate["metric"] == metric_name].copy()
        fig, ax = plt.subplots(figsize=(9.5, 5.4))

        for probe in PROBES:
            probe_table = (
                subset.loc[subset["probe"] == probe]
                .set_index("combination")
                .reindex(COMBINATION_ORDER)
            )
            if probe_table["mean_delta"].isna().any():
                raise ValueError(f"Missing paired delta rows for {metric_name!r}, probe={probe!r}")

            ax.errorbar(
                FIFTH_CHANNEL_ORDER_HZ,
                probe_table["mean_delta"].to_numpy(dtype=float),
                yerr=probe_table["sd_delta"].fillna(0.0).to_numpy(dtype=float),
                marker="o",
                capsize=4,
                label=probe,
            )

        ax.axhline(0.0, linestyle="--", linewidth=1.2)
        ax.axvline(REFERENCE_FIFTH_CHANNEL_HZ, linestyle=":", linewidth=1.0)
        _configure_frequency_axis(ax)
        ax.set_ylabel(f"Δ {metric_name} vs {REFERENCE_FIFTH_CHANNEL_HZ:g} Hz")
        ax.set_title(
            f"Paired frequency effect: {metric_name} relative to "
            f"{REFERENCE_FIFTH_CHANNEL_HZ:g} Hz"
        )
        ax.grid(alpha=0.25)
        ax.legend(title="CNN probe")
        fig.tight_layout()

        path = output_dir / f"delta_{safe_slug(metric_name)}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[metric_name] = path

    return saved


if not METRIC_AGGREGATE.empty:
    METRIC_FIGURES = plot_selected_metric_comparisons(METRIC_AGGREGATE)
    DELTA_FIGURES = plot_paired_delta_comparisons(PAIRED_DELTA_AGGREGATE)
    display(
        pd.DataFrame(
            {
                "metric": list(METRIC_FIGURES),
                "absolute_figure": list(METRIC_FIGURES.values()),
                "paired_delta_figure": [DELTA_FIGURES[name] for name in METRIC_FIGURES],
            }
        )
    )

## 7. Aggregate confusion matrices across seeds

For each completed run, `embeddings_test.npz` provides true labels `y` and CNN predictions `cnn_pred`.

For each seed the notebook computes a **row-normalized confusion matrix**:

\[
C_{ij} = P(\hat y=j \mid y=i)
\]

Then, for each **fifth frequency × probe**, it computes the element-wise mean and SD across the five training seeds.

The diagonal is per-class recall; off-diagonal cells are conditional confusion rates. Because the split and canonical cohort are fixed, matrices are directly paired across frequency conditions.

In [ ]:
IDX_TO_CLASS = {index: label for label, index in FIXED_SPLIT.class_to_idx.items()}
CLASS_LABELS = [IDX_TO_CLASS[index] for index in range(len(IDX_TO_CLASS))]
NUM_CLASSES = len(CLASS_LABELS)


def row_normalized_confusion(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}")

    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (y_true, y_pred), 1)
    row_totals = counts.sum(axis=1, keepdims=True)
    if np.any(row_totals == 0):
        empty_classes = np.flatnonzero(row_totals[:, 0] == 0).tolist()
        raise ValueError(f"Test split has zero samples for class indices: {empty_classes}")
    return counts / row_totals


def load_test_predictions(run_dir: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    path = run_dir / "embeddings_test.npz"
    if not path.is_file():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as payload:
        required = {"y", "cnn_pred", "sample_id"}
        missing = sorted(required.difference(payload.files))
        if missing:
            raise KeyError(f"{path} is missing arrays: {missing}")
        return (
            payload["y"].astype(np.int64, copy=False),
            payload["cnn_pred"].astype(np.int64, copy=False),
            payload["sample_id"].astype(str),
        )


def aggregate_confusions() -> tuple[dict[tuple[str, str], dict[str, np.ndarray]], pd.DataFrame]:
    aggregate: dict[tuple[str, str], dict[str, np.ndarray]] = {}
    recall_rows: list[dict[str, object]] = []

    for combination in COMBINATIONS:
        for probe in PROBES:
            matrices: list[np.ndarray] = []
            reference_ids: np.ndarray | None = None
            reference_y: np.ndarray | None = None

            for seed in TRAINING_SEEDS:
                run_dir = expected_run_dir(combination, seed, probe)
                y_true, y_pred, sample_ids = load_test_predictions(run_dir)

                # Seeds must evaluate the same fixed test cohort in the same order.
                if reference_ids is None:
                    reference_ids = sample_ids
                    reference_y = y_true
                else:
                    if not np.array_equal(reference_ids, sample_ids):
                        raise AssertionError(
                            f"Test sample ordering changed across seeds for {combination}, {probe}"
                        )
                    if not np.array_equal(reference_y, y_true):
                        raise AssertionError(
                            f"Test labels changed across seeds for {combination}, {probe}"
                        )

                matrix = row_normalized_confusion(y_true, y_pred, NUM_CLASSES)
                matrices.append(matrix)

                for class_index, class_label in enumerate(CLASS_LABELS):
                    recall_rows.append(
                        {
                            "combination": combination,
                            "fifth_channel_hz": FIFTH_CHANNEL_HZ_BY_COMBINATION[combination],
                            "probe": probe,
                            "training_seed": int(seed),
                            "class_index": class_index,
                            "class_label": class_label,
                            "recall": float(matrix[class_index, class_index]),
                        }
                    )

            stack = np.stack(matrices, axis=0)
            aggregate[(combination, probe)] = {
                "mean": stack.mean(axis=0),
                "sd": stack.std(axis=0, ddof=1),
                "all": stack,
            }

    recall_table = pd.DataFrame(recall_rows)
    recall_table.to_csv(OUTPUT_ROOT / "per_class_recall_by_seed.csv", index=False)
    return aggregate, recall_table


if not MASTER_RESULTS.empty:
    CONFUSION_AGGREGATE, PER_CLASS_RECALL = aggregate_confusions()
    display(PER_CLASS_RECALL.head())
else:
    CONFUSION_AGGREGATE = {}
    PER_CLASS_RECALL = pd.DataFrame()

## 8. Confusion matrices

Two complementary outputs are produced from the seed-averaged row-normalized matrices:

1. **One figure per fifth-frequency condition.** Each figure contains `cnn_s`, `cnn_m`, and `cnn_l` side by side. This guarantees that every user-specified combination has its own confusion-matrix figure.
2. **Cross-frequency small multiples per CNN probe.** These make it easy to compare how the same probe's confusion pattern changes with the fifth wavelet frequency.

All mean confusion matrices use the same class order and 0–1 scale. Optional SD small multiples remain available as a stability diagnostic but are disabled by default.

In [ ]:
def _subplot_grid(n: int) -> tuple[int, int]:
    cols = min(3, max(1, n))
    rows = math.ceil(n / cols)
    return rows, cols


def _annotate_confusion(ax: plt.Axes, matrix: np.ndarray) -> None:
    if NUM_CLASSES > ANNOTATE_CONFUSION_IF_CLASSES_LEQ:
        return
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)


def plot_confusion_by_combination(
    aggregate: Mapping[tuple[str, str], Mapping[str, np.ndarray]],
    *,
    output_dir: Path = FIGURE_ROOT / "confusion_by_frequency",
) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    saved: dict[str, Path] = {}

    for combination in COMBINATION_ORDER:
        frequency_hz = FIFTH_CHANNEL_HZ_BY_COMBINATION[combination]
        fig, axes = plt.subplots(
            1,
            len(PROBES),
            figsize=(5.0 * len(PROBES), 4.8),
            squeeze=False,
            constrained_layout=True,
        )
        image = None
        for probe_index, probe in enumerate(PROBES):
            ax = axes[0][probe_index]
            matrix = np.asarray(aggregate[(combination, probe)]["mean"], dtype=float)
            image = ax.imshow(matrix, vmin=0.0, vmax=1.0, aspect="auto")
            ax.set_title(probe)
            ax.set_xlabel("Predicted class")
            ax.set_ylabel("True class")
            ax.set_xticks(np.arange(NUM_CLASSES))
            ax.set_yticks(np.arange(NUM_CLASSES))
            ax.set_xticklabels(CLASS_LABELS, rotation=90)
            ax.set_yticklabels(CLASS_LABELS)
            _annotate_confusion(ax, matrix)

        fig.suptitle(
            f"Fifth channel = {frequency_hz:g} Hz | fixed backbone = {FIXED_WAVELET_CHANNELS_HZ} Hz\n"
            f"Mean row-normalized confusion across {len(TRAINING_SEEDS)} training seeds"
        )
        if image is not None:
            fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.80, label="conditional rate")

        path = output_dir / f"confusion_{safe_slug(combination)}_mean.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[combination] = path

    return saved


def plot_confusion_small_multiples(
    aggregate: Mapping[tuple[str, str], Mapping[str, np.ndarray]],
    *,
    statistic: str = "mean",
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if statistic not in {"mean", "sd"}:
        raise ValueError("statistic must be 'mean' or 'sd'")

    saved: dict[str, Path] = {}
    combinations = COMBINATION_ORDER

    if statistic == "mean":
        vmin, vmax = 0.0, 1.0
    else:
        max_sd = max(
            float(np.nanmax(aggregate[(combination, probe)]["sd"]))
            for combination in combinations
            for probe in PROBES
        )
        vmin, vmax = 0.0, max(max_sd, 1e-12)

    for probe in PROBES:
        rows, cols = _subplot_grid(len(combinations))
        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(5.1 * cols, 4.7 * rows),
            squeeze=False,
            constrained_layout=True,
        )
        image = None

        for index, combination in enumerate(combinations):
            ax = axes[index // cols][index % cols]
            matrix = np.asarray(aggregate[(combination, probe)][statistic], dtype=float)
            image = ax.imshow(matrix, vmin=vmin, vmax=vmax, aspect="auto")
            ax.set_title(f"fifth = {FIFTH_CHANNEL_HZ_BY_COMBINATION[combination]:g} Hz")
            ax.set_xlabel("Predicted class")
            ax.set_ylabel("True class")
            ax.set_xticks(np.arange(NUM_CLASSES))
            ax.set_yticks(np.arange(NUM_CLASSES))
            ax.set_xticklabels(CLASS_LABELS, rotation=90)
            ax.set_yticklabels(CLASS_LABELS)
            if statistic == "mean":
                _annotate_confusion(ax, matrix)

        for index in range(len(combinations), rows * cols):
            axes[index // cols][index % cols].axis("off")

        title = (
            f"{probe}: mean row-normalized confusion across {len(TRAINING_SEEDS)} seeds"
            if statistic == "mean"
            else f"{probe}: row-normalized confusion SD across {len(TRAINING_SEEDS)} seeds"
        )
        fig.suptitle(title)
        if image is not None:
            fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.78, label=statistic)

        path = output_dir / f"confusion_small_multiples_{probe}_{statistic}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path

    return saved


if CONFUSION_AGGREGATE:
    CONFUSION_BY_FREQUENCY_FIGURES = plot_confusion_by_combination(CONFUSION_AGGREGATE)
    CONFUSION_MEAN_FIGURES = plot_confusion_small_multiples(
        CONFUSION_AGGREGATE, statistic="mean"
    )
    if PLOT_CONFUSION_STD:
        CONFUSION_SD_FIGURES = plot_confusion_small_multiples(
            CONFUSION_AGGREGATE, statistic="sd"
        )

## 9. Per-class recall heatmaps

The diagonal of each row-normalized confusion matrix is class recall.

For every CNN probe the notebook builds a heatmap with:

- **rows** = classes;
- **columns** = automatically inferred fifth wavelet frequencies, sorted numerically;
- **cell value** = mean recall across the five training seeds.

A second heatmap shows recall SD across seeds. For a dense frequency sweep, this is the main class-level view for identifying which classes benefit from low-, mid-, or high-frequency fifth channels.

In [ ]:
def aggregate_per_class_recall(table: pd.DataFrame) -> pd.DataFrame:
    if table.empty:
        raise RuntimeError("No per-class recall records available")
    return (
        table
        .groupby(["probe", "combination", "class_index", "class_label"], as_index=False)
        .agg(
            mean_recall=("recall", "mean"),
            sd_recall=("recall", "std"),
            n=("recall", "count"),
        )
    )


def plot_per_class_recall_heatmaps(
    aggregate: pd.DataFrame,
    *,
    statistic: str = "mean_recall",
    output_dir: Path = FIGURE_ROOT,
) -> dict[str, Path]:
    if statistic not in {"mean_recall", "sd_recall"}:
        raise ValueError("statistic must be 'mean_recall' or 'sd_recall'")

    saved: dict[str, Path] = {}
    combination_order = COMBINATION_ORDER

    if statistic == "mean_recall":
        vmin, vmax = 0.0, 1.0
    else:
        max_sd = float(aggregate["sd_recall"].max())
        vmin, vmax = 0.0, max(max_sd, 1e-12)

    for probe in PROBES:
        subset = aggregate.loc[aggregate["probe"] == probe].copy()
        pivot = (
            subset.pivot(index="class_label", columns="combination", values=statistic)
            .reindex(index=CLASS_LABELS, columns=combination_order)
        )
        if pivot.isna().any().any():
            raise ValueError(f"Missing per-class recall cells for probe={probe}, statistic={statistic}")

        fig, ax = plt.subplots(
            figsize=(max(8, 1.35 * len(combination_order)), max(5, 0.42 * NUM_CLASSES + 2.5))
        )
        image = ax.imshow(pivot.to_numpy(dtype=float), vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(np.arange(len(combination_order)))
        ax.set_xticklabels(combination_order, rotation=30, ha="right")
        ax.set_yticks(np.arange(NUM_CLASSES))
        ax.set_yticklabels(CLASS_LABELS)
        ax.set_xlabel("Fifth wavelet frequency (Hz)")
        ax.set_ylabel("Class")
        ax.set_title(
            f"{probe}: per-class recall by fifth wavelet frequency "
            + ("mean across training seeds" if statistic == "mean_recall" else "SD across training seeds")
        )
        fig.colorbar(image, ax=ax, label=statistic)

        if NUM_CLASSES <= ANNOTATE_CONFUSION_IF_CLASSES_LEQ and len(combination_order) <= 8:
            values = pivot.to_numpy(dtype=float)
            for i in range(values.shape[0]):
                for j in range(values.shape[1]):
                    ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=7)

        fig.tight_layout()
        path = output_dir / f"per_class_recall_{probe}_{statistic}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path

    return saved


if not PER_CLASS_RECALL.empty:
    PER_CLASS_RECALL_AGG = aggregate_per_class_recall(PER_CLASS_RECALL)
    PER_CLASS_RECALL_AGG.to_csv(OUTPUT_ROOT / "per_class_recall_aggregate.csv", index=False)
    display(PER_CLASS_RECALL_AGG.head(20))

    RECALL_MEAN_FIGURES = plot_per_class_recall_heatmaps(
        PER_CLASS_RECALL_AGG,
        statistic="mean_recall",
    )
    RECALL_SD_FIGURES = plot_per_class_recall_heatmaps(
        PER_CLASS_RECALL_AGG,
        statistic="sd_recall",
    )

## 10. Compact report tables

This section creates presentation-friendly tables for:

- main selected metrics (`mean ± SD` across training seeds);
- seed-paired deltas vs the reference fifth frequency;
- supplementary diagnostics;
- per-class recall.

The paired-delta table is especially useful when the research question is the marginal utility of replacing the fifth wavelet channel while keeping `(1,2,4,8)` fixed.

In [ ]:
def format_mean_sd(mean: float, sd: float, digits: int = 3) -> str:
    sd_value = 0.0 if pd.isna(sd) else float(sd)
    return f"{float(mean):.{digits}f} ± {sd_value:.{digits}f}"


if not METRIC_AGGREGATE.empty:
    formatted = METRIC_AGGREGATE.copy()
    formatted["mean ± SD"] = [
        format_mean_sd(mean, sd)
        for mean, sd in zip(formatted["mean"], formatted["sd"])
    ]
    METRIC_REPORT_TABLE = formatted.pivot_table(
        index=["metric", "fifth_channel_hz", "combination"],
        columns="probe",
        values="mean ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    METRIC_REPORT_TABLE.to_csv(OUTPUT_ROOT / "selected_metric_report_table.csv")
    display(METRIC_REPORT_TABLE)

if not PAIRED_DELTA_AGGREGATE.empty:
    delta_formatted = PAIRED_DELTA_AGGREGATE.copy()
    delta_formatted["mean Δ ± SD"] = [
        format_mean_sd(mean, sd)
        for mean, sd in zip(delta_formatted["mean_delta"], delta_formatted["sd_delta"])
    ]
    PAIRED_DELTA_REPORT_TABLE = delta_formatted.pivot_table(
        index=["metric", "fifth_channel_hz", "combination"],
        columns="probe",
        values="mean Δ ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    PAIRED_DELTA_REPORT_TABLE.to_csv(OUTPUT_ROOT / "paired_delta_report_table.csv")
    display(PAIRED_DELTA_REPORT_TABLE)

if not SUPPLEMENTARY_METRIC_AGGREGATE.empty:
    supplementary_formatted = SUPPLEMENTARY_METRIC_AGGREGATE.copy()
    supplementary_formatted["mean ± SD"] = [
        format_mean_sd(mean, sd)
        for mean, sd in zip(
            supplementary_formatted["mean"], supplementary_formatted["sd"]
        )
    ]
    SUPPLEMENTARY_METRIC_REPORT_TABLE = supplementary_formatted.pivot_table(
        index=["metric", "fifth_channel_hz", "combination"],
        columns="probe",
        values="mean ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    SUPPLEMENTARY_METRIC_REPORT_TABLE.to_csv(
        OUTPUT_ROOT / "supplementary_metric_report_table.csv"
    )
    display(SUPPLEMENTARY_METRIC_REPORT_TABLE)

if 'PER_CLASS_RECALL_AGG' in globals() and not PER_CLASS_RECALL_AGG.empty:
    recall_formatted = PER_CLASS_RECALL_AGG.copy()
    recall_formatted["mean ± SD"] = [
        format_mean_sd(mean, sd)
        for mean, sd in zip(
            recall_formatted["mean_recall"], recall_formatted["sd_recall"]
        )
    ]
    PER_CLASS_RECALL_REPORT = recall_formatted.loc[
        :, ["probe", "combination", "class_label", "mean ± SD"]
    ]
    PER_CLASS_RECALL_REPORT.to_csv(
        OUTPUT_ROOT / "per_class_recall_report_table.csv",
        index=False,
    )
    display(PER_CLASS_RECALL_REPORT.head(30))

## 11. Output checklist

After a complete benchmark, the top-level output directory should contain:

```text
benchmark_design.json
cross_combination_preflight.csv
master_results.csv
selected_metric_aggregate.csv
selected_metric_report_table.csv
supplementary_metric_aggregate.csv
supplementary_metric_report_table.csv
paired_delta_by_seed.csv
paired_delta_aggregate.csv
paired_delta_report_table.csv
per_class_recall_by_seed.csv
per_class_recall_aggregate.csv
per_class_recall_report_table.csv
comparison_figures/
    metric_*.png                         # absolute mean ± SD frequency response
    delta_*.png                          # seed-paired Δ vs reference frequency
    confusion_by_frequency/
        confusion_0.5Hz_mean.png         # one 3-probe figure per condition
        confusion_<fifth>Hz_mean.png
    confusion_small_multiples_cnn_s_mean.png
    confusion_small_multiples_cnn_m_mean.png
    confusion_small_multiples_cnn_l_mean.png
    confusion_small_multiples_*_sd.png   # optional; disabled by default
    per_class_recall_cnn_s_mean_recall.png
    per_class_recall_cnn_m_mean_recall.png
    per_class_recall_cnn_l_mean_recall.png
    per_class_recall_*_sd_recall.png
<fifth>Hz/seed_<seed>/<probe>/
    summary.csv
    classification_splits.csv
    training_history.csv
    embeddings_test.npz
    ... standard Experiment C artifacts ...
```

### Recommended interpretation

Treat the evidence in this order:

1. **CNN balanced accuracy** as the primary task metric.
2. **CNN macro-F1** as the complementary task metric.
3. **kNN BA, linear-probe BA, and retrieval macro-mAP** as evidence that the representation itself becomes more task-readable.
4. **Per-class recall and confusion matrices** to identify class-specific frequency effects.
5. **SameLabel@1 and geometry metrics** only as supporting diagnostics.

Report every probe separately. Error bars are training-seed SD, not user-split uncertainty. For marginal frequency claims, prioritize the **seed-paired Δ vs the 0.5 Hz reference** over differences between independently summarized means.

When conditions came from the sweep launcher, keep `experiment_c_frequency_benchmark_inputs.json` beside the generated dataset roots as the provenance map from fifth frequency to Action0/Action1 inputs.
